In [0]:
silver_df = spark.table("workspace.telecom_silver.network_events")
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable("workspace.telecom_silver.network_events_partitioned")

display(spark.sql("""
DESCRIBE DETAIL workspace.telecom_silver.network_events_partitioned
"""))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,199fe688-e536-4da8-8894-9aa228076ca5,workspace.telecom_silver.network_events_partitioned,null,,2026-06-14T03:37:37.677Z,2026-06-14T03:37:53.000Z,List(event_date),List(),8,1779587,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(spark.sql("""
SELECT region, network_type, COUNT(*) AS total_events
FROM workspace.telecom_silver.network_events_partitioned
WHERE event_date = '2026-06-09'
GROUP BY region, network_type
"""))

region,network_type,total_events
Chicago,4G,350
Houston,5G,370
Chicago,LTE,395
New York,LTE,372
Austin,5G,386
Dallas,4G,357
Dallas,5G,426
Atlanta,5G,407
Chicago,5G,393
Austin,LTE,381


In [0]:
spark.sql("""
SELECT region, network_type, COUNT(*) AS total_events
FROM workspace.telecom_silver.network_events_partitioned
WHERE event_date = '2026-06-09'
GROUP BY region, network_type
""").explain(True)

== Parsed Logical Plan ==
'Aggregate ['region, 'network_type], ['region, 'network_type, 'COUNT(1) AS total_events#12351]
+- 'Filter ('event_date = 2026-06-09)
   +- 'UnresolvedRelation [workspace, telecom_silver, network_events_partitioned], [], false

== Analyzed Logical Plan ==
region: string, network_type: string, total_events: bigint
Aggregate [region#12369, network_type#12370], [region#12369, network_type#12370, count(1) AS total_events#12351L]
+- Filter (event_date#12379 = cast(2026-06-09 as date))
   +- SubqueryAlias workspace.telecom_silver.network_events_partitioned
      +- Relation workspace.telecom_silver.network_events_partitioned[event_id#12366,customer_id#12367,cell_tower_id#12368,region#12369,network_type#12370,device_type#12371,event_type#12372,event_timestamp#12373,signal_strength#12374,latency_ms#12375,dropped_call#12376,data_usage_mb#12377,network_quality#12378,event_date#12379] parquet

== Optimized Logical Plan ==
Aggregate [region#12369, network_type#12370], [reg

In [0]:
valid_count = spark.table(
    "workspace.telecom_silver.network_events"
).count()

rejected_count = spark.table(
    "workspace.telecom_silver.network_events_rejected"
).count()

quality_summary_df = spark.createDataFrame(
    [
        ("valid_records", valid_count),
        ("rejected_records", rejected_count)
    ],
    ["metric_name", "record_count"]
)

display(quality_summary_df)

quality_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.telecom_gold.data_quality_summary"
    )
display(
    spark.table(
        "workspace.telecom_gold.data_quality_summary"
    )
)

metric_name,record_count
valid_records,50000
rejected_records,0


metric_name,record_count
valid_records,50000
rejected_records,0
